# Download BDD100K Dataset

Downloads the BDD100K 2D detection subset to `/Volumes/T7/data-engine-dataset` via Kaggle.

**Before running:**
1. Create a free [Kaggle](https://www.kaggle.com) account
2. Go to Settings > API > Create New Token — this downloads a `kaggle.json` file
3. Move it to `~/.kaggle/kaggle.json` and `chmod 600 ~/.kaggle/kaggle.json`

In [2]:
from pathlib import Path

DATA_ROOT = Path("/Volumes/T7/data-engine-dataset")
DOWNLOAD_DIR = DATA_ROOT / "downloads"
EXTRACT_DIR = DATA_ROOT / "bdd100k"

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Download dir: {DOWNLOAD_DIR}")
print(f"Extract dir:  {EXTRACT_DIR}")

Download dir: /Volumes/T7/data-engine-dataset/downloads
Extract dir:  /Volumes/T7/data-engine-dataset/bdd100k


## Step 1: Install Kaggle CLI

In [3]:
!pip install kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [kaggle]━━━━ 1/3 [jupytext]


## Step 2: Verify Kaggle credentials

In [5]:
kaggle_creds = Path.home() / ".kaggle" / "kaggle.json"
assert kaggle_creds.exists(), (
    f"Missing {kaggle_creds}. Download your API token from "
    "https://www.kaggle.com/settings > API > Create New Token"
)
print(f"Kaggle credentials found: {kaggle_creds}")

Kaggle credentials found: /Users/Dylan/.kaggle/kaggle.json


## Step 3: Download from Kaggle

Dataset: [awsaf49/bdd100k-dataset](https://www.kaggle.com/datasets/awsaf49/bdd100k-dataset) (~6.8 GB)

In [6]:
import subprocess

subprocess.run(
    [
        "kaggle", "datasets", "download",
        "-d", "awsaf49/bdd100k-dataset",
        "-p", str(DOWNLOAD_DIR),
    ],
    check=True,
)
print("Download complete.")

Dataset URL: https://www.kaggle.com/datasets/awsaf49/bdd100k-dataset
License(s): CC0-1.0


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉| 6.38G/6.38G [02:31<00:00, 54.7MB/s]


Download complete.


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6.38G/6.38G [02:31<00:00, 45.3MB/s]


## Step 4: Extract

In [7]:
import zipfile

zip_path = DOWNLOAD_DIR / "bdd100k-dataset.zip"
assert zip_path.exists(), f"Expected zip at {zip_path}"

print(f"Extracting {zip_path.name} ({zip_path.stat().st_size / 1e9:.1f} GB)...")
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(EXTRACT_DIR)
print("Done.")

Extracting bdd100k-dataset.zip (6.9 GB)...
Done.


## Step 5: Verify

Check that the expected directories and file counts are present. The exact paths may vary depending on how the Kaggle upload is structured — adjust if needed.

In [8]:
import os

# Print the top-level structure so we can see what we got
for root, dirs, files in os.walk(EXTRACT_DIR):
    depth = root.replace(str(EXTRACT_DIR), "").count(os.sep)
    if depth < 3:
        indent = "  " * depth
        print(f"{indent}{Path(root).name}/")
        if files:
            print(f"{indent}  ({len(files)} files)")

bdd100k/
  (2 files)
  bdd100k/
    bdd100k/
  labels/
    (2 files)


In [9]:
# Count images — adjust these paths based on the output above
# Common structures from Kaggle uploads:
#   bdd100k/images/100k/train/  OR  images/100k/train/

for candidate_root in [EXTRACT_DIR, EXTRACT_DIR / "bdd100k"]:
    train_dir = candidate_root / "images" / "100k" / "train"
    val_dir = candidate_root / "images" / "100k" / "val"
    if train_dir.exists():
        n_train = len(list(train_dir.glob("*.jpg")))
        n_val = len(list(val_dir.glob("*.jpg"))) if val_dir.exists() else 0
        print(f"Found images at: {candidate_root}")
        print(f"  Train: {n_train:,} images (expected ~70,000)")
        print(f"  Val:   {n_val:,} images (expected ~10,000)")
        break
else:
    print("Could not find images directory. Check the tree output above and adjust paths.")

Could not find images directory. Check the tree output above and adjust paths.


## Step 6: Symlink into project

Creates a symlink from the project's `data/raw/bdd100k` to the T7 drive so pipeline code can reference `data/raw/bdd100k/` without caring where the data physically lives.

In [10]:
PROJECT_RAW = Path("/Users/Dylan/Documents/data_engine/data/raw")
PROJECT_RAW.mkdir(parents=True, exist_ok=True)

# Point to whichever root contains images/ and labels/
# Adjust this if the Kaggle extract has a different structure
LINK_TARGET = EXTRACT_DIR
for candidate in [EXTRACT_DIR / "bdd100k", EXTRACT_DIR]:
    if (candidate / "images").exists():
        LINK_TARGET = candidate
        break

symlink = PROJECT_RAW / "bdd100k"

if symlink.exists() or symlink.is_symlink():
    print(f"Symlink already exists: {symlink} -> {symlink.resolve()}")
else:
    symlink.symlink_to(LINK_TARGET)
    print(f"Created symlink: {symlink} -> {LINK_TARGET}")

Created symlink: /Users/Dylan/Documents/data_engine/data/raw/bdd100k -> /Volumes/T7/data-engine-dataset/bdd100k
